In [2]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error

# 1) Load data
df = pd.read_csv("../Capstone/Capstone Dataset (New).csv")

X = df.drop(
    ["Array type", "Predicted angular insertion depth (deg)",
     "Actual angular insertion depth (deg)"],
    axis=1
)
y = df["Actual angular insertion depth (deg)"]

# 2) Define model and search space
rf = RandomForestRegressor(random_state=42)

param_grid = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 5],
    "max_features": ["auto", "sqrt", "log2"]
}

# 3) Grid search
gs = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring="neg_mean_absolute_error",
    cv=5,       # change to LeaveOneOut() if dataset is very small
    n_jobs=-1,
    verbose=1,
    refit=True
)
gs.fit(X, y)

print("Best params:", gs.best_params_)
print("CV best MAE:", -gs.best_score_)

Fitting 5 folds for each of 432 candidates, totalling 2160 fits


C:\Users\shell\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:540: FitFailedWarning: 
720 fits failed out of a total of 2160.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
332 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\shell\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\shell\anaconda3\Lib\site-packages\sklearn\base.py", line 1466, in wrapper
    estimator._validate_params()
  File "C:\Users\shell\anaconda3\Lib\site-packages\sklearn\base.py", line 666, in _validate_params
    validate_parameter_constraints(
  File "C:\Users\shell\anaconda3\Lib\site-packa

Best params: {'max_depth': 5, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 100}
CV best MAE: 44.51690970443481


In [3]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import mean_absolute_error

# 1) Load dataset
df = pd.read_csv("Capstone Dataset (New).csv")

# 2) Separate features (X) and target (y)
X = df.drop(
    ["Array type", "Predicted angular insertion depth (deg)",
     "Actual angular insertion depth (deg)"],
    axis=1
)
y = df["Actual angular insertion depth (deg)"]

# 3) Leave-One-Out Method
loo = LeaveOneOut()

y_true_all = []
y_pred_all = []

for train_index, test_index in loo.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Random Forest Regressor (Using best parameters)
    rf = RandomForestRegressor(
        n_estimators=100,
        max_depth=5,
        max_features="sqrt",
        min_samples_split=2,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )

    # Fit model
    rf.fit(X_train, y_train)

    # Predict the held-out sample
    y_pred = rf.predict(X_test)[0]

    # Collect predictions and true values
    y_pred_all.append(y_pred)
    y_true_all.append(y_test.values[0])

# 4) Evaluation (overall MAE across all LOO folds)
mae = mean_absolute_error(y_true_all, y_pred_all)
print("MAE:", mae)

MAE: 38.51974997823152
